In [88]:
import numpy as np
import pyvisa

In [2]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'GPIB0::23::INSTR')


In [3]:
import pyvisa
class weeder:
    # command string structure: header character ("A") + command + carriage return ("\r")
    # header characters in Table 1, command list in Table 2
    # commands are echoed back after successful execution in the same format as received
    # I'm not sure yet if spaces are required to separate the three parts of the command string
    # diagnostic characters: (?) for invalid command or variable, (!) for power-on reset or brown-out
    #header = "A" # could define header at the beginning to avoid having to redefine later on...

    # defines and discovers stepper motor
    def __init__(self, visa_name, timeout = 5000):
        rm = pyvisa.ResourceManager()
        self.pyvisa = rm.open_resource(visa_name, baud_rate = 9600, data_bits = 8, 
                                       parity = pyvisa.constants.Parity.none, 
                                       stop_bits = pyvisa.constants.StopBits.one)
        self.pyvisa.timeout = 5000 
       
    # moves stepper motor to a specific position 
    # position = 0 to 2^24 - 1 = 16,777,215
    def move(self, header: str, position: int):
        self.pyvisa.write(header + "M" + str(position) + "\r")
        return self.pyvisa.read("\r")

    # moves stepper motor in a specific direction until runoff counter is triggered
    # direction = + or -
    # runoff = 0 to 255 (if omitted, defaults to zero)
    # I don't think we'll use this one
    def home(self, header: str, direction: str, runoff = 0):
        self.pyvisa.write(header + "H" + direction + str(runoff) + "\r")
        return self.pyvisa.read("\r")

    # moves stepper motor in a continuous rotation in a specific direction 
    # velocity can be changed during motor rotation
    # direction = + or - (if omitted, rotation is halted)
    # I don't think we'll use this one
    def drive(self, header: str, direction = "+"):
        self.pyvisa.write(header + "D" + direction + "\r")
        return self.pyvisa.read("\r")

    # moves stepper motor one step in a specific direction
    # direction = + or -
    def step (self, header: str, direction: str):
        self.pyvisa.write(header + "S" + direction + "\r")
        return self.pyvisa.read("\r")

    # sets the pulse-per-second (pps) rate for move, home, and drive functions
    # velocity = 1 to 200 (will be multiplied by 50 pps; if omitted, will return current setting)
    def velocity(self, header: str, velocity = 1, query = False):
        if query:
            self.pyvisa.write(header + "V\r")
            return self.pyvisa.read("\r")
        else:
            self.pyvisa.write(header + "V" + str(velocity) + "\r")
        
    # sets the ramp rate used in the acceleration and deceleration curves
    # rate = 1 to 255 (default = 50; if omitted, will return current value)
    def ramp_rate(self, header: str, rate = 50, query = False):
        if query:
            self.pyvisa.write(header + "R\r")
            return self.pyvisa.read("\r")
        else:
            self.pyvisa.write(header + "R" + str(rate) + "\r")
        
    # modifies motor position counter
    # position = 0 to 16,777,215 (if omitted, will read current position)
    def position(self, header: str, position = 0, query = False):
        if query:
            self.pyvisa.write(header + "P\r")
            return self.pyvisa.read("\r")
        else:
            self.pyvisa.write(header + "P" + str(position) + "\r")
        
    # sets the excitation mode to determine the number of microsteps per step
    # microsteps = 1, 2, 4, 8, 16, 32, or 64 (default = 1; if omitted, will return current setting)
    def excite(self, header: str, microsteps = 1, query = False):
        if query:
            self.pyvisa.write(header + "E\r")
            return self.pyvisa.read("\r") 
        else:
            self.pyvisa.write(header + "E" + str(microsteps) + "\r")

    # sets the drive current, used when the motor is rotating
    # current = 1 to 20 (will be multiplied by 0.1 A; default = 5; if omitted, will return present value)
    def current(self, header: str, current = 5, query = False):
        if query:
            self.pyvisa.write(header + "C\r")
            return self.pyvisa.read("\r")
        else:
            self.pyvisa.write(header + "C" + str(current) + "\r")
        
    # sets the idle reduction current, used when the motor is at rest
    # idle = 1 to 10 (will be multipled by 10 and read as % of drive current; default = 2)
    # if idle is omitted, will return the present value
    def idle(self, header: str, idle = 2, query = False):
        if query:
            self.pyvisa.write(header + "I\r")
            return self.pyvisa.read("\r")
        else:
            self.pyvisa.write(header + "I" + str(idle) + "\r")
        
    # reads the current state of the limit switch indicated by direction (+ or -) 
    # returns "direction C" for closed and "direction O" for open
    # this is a newer function not mentioned in our copy of the manual, so I'm not sure if this will work 
    def limit(self, header: str, direction: str):
        self.pyvisa.write(header + "L" + direction + "\r")
        return self.pyvisa.read("\r")

    # write motor settings to txt files
    # motor position will always be saved, but optionally one can save other parameters as well
    def save(self, header: str, query = True, more = False):
        # query motor settings
        current_position = self.position(header = "A", query = True)[1:]
        velocity = self.velocity(header = "A", query = True)[2:]
        ramp_rate = self.ramp_rate(header = "A", query = True)[2:]
        excitation_mode = self.excite(header = "A", query = True)[2:]
        drive_current = self.current(header = "A", query = True)[2:]
        idle_current = self.idle(header = "A", query = True)[2:]

        # save the motor position to file
        with open(r"X:\migratedData\Data\motor1.txt",'w') as file:
                file.write(str(current_position))
            
        # used if you want to save other motor settings
        if more:
            # save everything (self.function(args))
            with open(r"X:\migratedData\Data\motor-params.txt", 'w') as file:
                file.write("Velocity: " + str(velocity) + '\n')
                file.write("Ramp rate: " + str(ramp_rate) + '\n')
                file.write("Excitation mode: " + str(excitation_mode) + '\n')
                file.write("Drive current: " + str(drive_current) + '\n')
                file.write("Idle current: " + str(idle_current) + '\n')

        return "saving complete."

    # read out motor settings from txt file 
    def read(self, file_path: str):
        try:
            with open(file_path, 'r') as file:
                lines = file.readlines()
                for line in lines:
                    print(line.strip())
        except FileNotFoundError:
            print(f'Error: File not found at path: {file_path}')
            return None
        except Exception as e:
            print(f'An error occurred: {e}')
            return None

In [4]:
weeder = weeder("ASRL1::INSTR")

In [ ]:
weeder.save(header = "A", query = True, more = True)

In [ ]:
weeder.read(file_path = r"X:\migratedData\Data\motor1.txt")

In [ ]:
weeder.read(file_path = r"X:\migratedData\Data\motor-params.txt")

In [81]:
weeder.position(header = "A", position = 100)

In [85]:
weeder.position(header = "A",query = True)

'AP100'

In [87]:
weeder.velocity(header = "A",query = True)

'AV1'

In [ ]:
# I use string slicing to remove the header and command characters at the beginning and leave the numerical value behind
weeder.position(header = "A", position = 0, query = False)
current_position = weeder.position(header = "A", query=True)
print(f'Current position: {current_position[1:]}')

velocity = weeder.velocity(header = "A", query = True)
print(f'Velocity (pps): {velocity[2:]} x 50 pps')

ramp_rate = weeder.ramp_rate(header = "A", query = True)
print(f'Ramp rate: {ramp_rate[2:]}')

excitation_mode = weeder.excite(header = "A", query = True)
print(f'Number of microsteps: {excitation_mode[2:]}')

drive_current = weeder.current(header = "A", query = True)
print(f'Drive current (A): {drive_current[2:]} x 0.1 A')

idle_current = weeder.idle(header = "A", query = True)
print(f'Idle reduction current (%): {idle_current[2:]} x 10%')

In [ ]:
weeder.save(header = "A", query = True, more = True)